# 07 - Classification And Decision Boundary

这一节把 06 的训练循环换成分类任务。

这一节只抓一句话：

> 模型输出一个 score，score 大于 0 判为正类，小于 0 判为负类；训练的目标是让正确类别的 score 离 0 更远。

你会看到：

```text
score -> sign(score) -> class label
score + label -> margin loss
训练后 -> decision boundary
```

<!-- xiao-preview:start -->
## 课前预习作业：先判断一个点属于哪一类

先不训练。给一个手写的线性分类器，算一次 score 和预测类别。

In [ ]:
# TODO: 把 None 改成你算出来的结果。
# score = w1*x1 + w2*x2 + b
# x = [2, -1], w = [1, 2], b = -1, true label y = -1
preview_score = None
preview_pred_label = None      # score > 0 填 1，否则填 -1
preview_margin = None          # margin = y * score


def qa_check_07_preview(score, pred_label, margin):
    values = [score, pred_label, margin]
    if any(v is None for v in values):
        print('请先填写 TODO：先算 score、预测类别和 margin。')
        return
    assert score == -1, f'score 应该是 -1，但你填的是 {score}'
    assert pred_label == -1, f'score < 0，所以预测类别应该是 -1，但你填的是 {pred_label}'
    assert margin == 1, f'margin = y * score = (-1) * (-1) = 1，但你填的是 {margin}'
    print('OK: 分类预习通过。')


qa_check_07_preview(preview_score, preview_pred_label, preview_margin)

<!-- xiao-preview:hint -->
<details>
<summary><strong>Show / Hide 提示</strong></summary>

先算：

```text
score = 1*2 + 2*(-1) + (-1)
```

如果 `score > 0`，预测 `1`；否则预测 `-1`。

</details>

<!-- xiao-preview:answer -->
<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
preview_score = -1
preview_pred_label = -1
preview_margin = 1
```

</details>

## 0. Setup

In [ ]:
from pathlib import Path
import sys
import random

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / 'micrograd', cwd.parent / 'micrograd']
PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / 'micrograd' / 'engine.py').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError(f'Could not find local micrograd package from {cwd}')

sys.path.insert(0, str(PROJECT_ROOT))

from micrograd.engine import Value
from micrograd.nn import MLP

random.seed(1337)
print('project root:', PROJECT_ROOT)

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception as exc:
    MATPLOTLIB_AVAILABLE = False
    print('matplotlib unavailable:', exc)

## 1. 分类不是直接输出类别，而是先输出 score

一个二分类模型可以先输出一个数字：

```text
score > 0 -> 预测 +1
score < 0 -> 预测 -1
```

这个 `score = 0` 的位置就是边界。二维平面里，它会变成一条线或一条弯曲边界。

In [ ]:
def label_from_score(score):
    return 1 if score > 0 else -1

for score in [-2.0, -0.1, 0.3, 4.0]:
    print(score, '->', label_from_score(score))

## 2. margin：不只要答对，还要答得有把握

如果真实标签 `y` 是 `1` 或 `-1`，我们看：

$$
margin = y \cdot score
$$

直觉：

| 情况 | margin | 意思 |
|---|---:|---|
| `y=1, score=2` | 2 | 答对，而且离边界远 |
| `y=1, score=0.2` | 0.2 | 答对，但很靠近边界 |
| `y=1, score=-1` | -1 | 答错 |

常用的 hinge loss 是：

$$
loss = max(0, 1 - margin)
$$

在 micrograd 里可以写成：

```python
(1 - y * score).relu()
```

In [ ]:
def hinge_loss(score, y):
    return (1 - y * score).relu()

examples = [
    (Value(2.0), 1),
    (Value(0.2), 1),
    (Value(-1.0), 1),
    (Value(-2.0), -1),
]

for score, y in examples:
    loss_value = hinge_loss(score, y)
    print(f'y={y:>2}, score={score.data:>4}, margin={y * score.data:>4}, loss={loss_value.data}')

## 3. 一个小分类数据集

这里每个点有两个输入 `x1, x2`，标签是 `1` 或 `-1`。

In [ ]:
xs = [
    [-2.0, -1.0],
    [-1.0, -2.0],
    [-2.0,  1.0],
    [ 1.0,  2.0],
    [ 2.0,  1.0],
    [ 2.0, -1.0],
]
ys = [-1, -1, -1, 1, 1, 1]

for x, y in zip(xs, ys):
    print(x, '->', y)

## 4. 训练一个 MLP 分类器

这里还是用 `MLP(2, [4, 1])`：

```text
输入 2 个数字 -> 隐藏层 4 个神经元 -> 输出 1 个 score
```

注意：输出层最后没有 ReLU，所以 score 可以是正数也可以是负数。

In [ ]:
random.seed(7)
model = MLP(2, [4, 1])
print(model)
print('parameters:', len(model.parameters()))


def predict_score(xrow):
    return model([Value(x) for x in xrow])


def classification_loss():
    scores = [predict_score(x) for x in xs]
    losses = [hinge_loss(score, y) for score, y in zip(scores, ys)]
    total = sum(losses) * (1.0 / len(losses))
    return total, scores

initial_loss, initial_scores = classification_loss()
print('initial loss:', initial_loss.data)
print('initial scores:', [round(s.data, 3) for s in initial_scores])

## 5. 跑训练循环

还是 06 那四步：

```text
forward -> zero_grad -> backward -> update
```

In [ ]:
learning_rate = 0.05
history = []

for step in range(80):
    total_loss, scores = classification_loss()
    history.append(total_loss.data)

    model.zero_grad()
    total_loss.backward()

    for p in model.parameters():
        p.data += -learning_rate * p.grad

    if step % 10 == 0 or step == 79:
        preds = [label_from_score(s.data) for s in scores]
        correct = sum(int(p == y) for p, y in zip(preds, ys))
        print(f'step {step:02d} loss={total_loss.data:.4f} acc={correct}/{len(ys)} preds={preds}')

final_loss, final_scores = classification_loss()
print('initial loss:', history[0])
print('final loss  :', final_loss.data)
assert final_loss.data < history[0]

## 6. 画出 decision boundary

`score = 0` 的地方就是分界线。下面只为了直觉看图，不需要深挖 matplotlib。

In [ ]:
if MATPLOTLIB_AVAILABLE:
    grid_x = [i / 10 for i in range(-30, 31)]
    grid_y = [i / 10 for i in range(-30, 31)]
    Z = []
    for yy in grid_y:
        row = []
        for xx in grid_x:
            row.append(predict_score([xx, yy]).data)
        Z.append(row)

    plt.figure(figsize=(5, 4))
    plt.contourf(grid_x, grid_y, Z, levels=[-999, 0, 999], alpha=0.25, colors=['#ff9999', '#99ccff'])
    plt.contour(grid_x, grid_y, Z, levels=[0], colors='black', linewidths=2)
    for x, y in zip(xs, ys):
        color = '#1f77b4' if y == 1 else '#d62728'
        marker = 'o' if y == 1 else 'x'
        plt.scatter(x[0], x[1], c=color, marker=marker, s=80)
    plt.title('Decision boundary: score = 0')
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.grid(True, alpha=0.2)
    plt.show()
else:
    print('matplotlib 不可用，跳过画图。')

## 7. TODO Lab - 写 hinge loss

TODO：补全一个函数。只要记住：

```text
loss = max(0, 1 - y * score)
```

在 micrograd 里用 ReLU 表示 `max(0, x)`。

In [ ]:
def student_hinge_loss(score, y):
    # TODO: 返回 (1 - y * score).relu()
    pass


def qa_check_07_hinge_loss(func):
    tests = [
        (Value(2.0), 1, 0.0),
        (Value(0.2), 1, 0.8),
        (Value(-0.5), -1, 0.5),
    ]
    for score, y, expected in tests:
        out = func(score, y)
        if out is None:
            print('请先填写 TODO：返回 hinge loss。')
            return
        assert abs(out.data - expected) < 1e-9, f'y={y}, score={score.data}: expected {expected}, got {out.data}'
    print('OK: hinge loss passed.')


qa_check_07_hinge_loss(student_hinge_loss)

<details>
<summary><strong>Show / Hide 提示</strong></summary>

`max(0, something)` 在 `Value` 里可以写成：

```python
something.relu()
```

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
def student_hinge_loss(score, y):
    return (1 - y * score).relu()
```

</details>

## What To Remember

这一节记住五句话：

```text
1. 分类模型先输出 score，不是直接输出文字类别。
2. score > 0 判为 +1，score < 0 判为 -1。
3. margin = y * score，margin 越大越有把握。
4. hinge loss = max(0, 1 - margin)。
5. decision boundary 就是 score = 0 的位置。
```

<!-- xiao-post:start -->
## 课后作业提交清单

```text
1. 我能解释 score、label、margin、hinge loss。
2. 我能写出 (1 - y * score).relu()。
3. 我跑过分类训练循环，并看到 loss 下降。
4. 我能说出 decision boundary 是 score = 0。
```

## AI 复盘检查 Prompt

```text
你是我的 micrograd 分类任务检查官。
我刚学完 score、margin、hinge loss 和 decision boundary。
请你一次只问一个问题，检查我是否理解：
1. score 和预测类别是什么关系
2. margin = y * score 的直觉
3. hinge loss = max(0, 1-margin) 为什么能惩罚分类错误
4. decision boundary 为什么是 score = 0
5. 训练循环和 06 的回归训练哪里一样、哪里不同
如果我答错，请用一个 score=0.2 或 score=-1 的小例子提示我。
```